# RQ3.1 — Input representation

Input: `../metrics/macro_f1.csv` (relative to this notebook; `results/rq3_1/metrics/macro_f1.csv` when run from the repository root).

Outputs (displayed inline; no files are written): descriptive statistics per input representation, normality and omnibus tests, Holm-corrected pairwise comparisons, and a pairwise table.

Run: *Run All*.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pingouin as pg
from IPython.display import display


SUBJECT = "project"

TREATMENTS = [
    "message_only",
    "diff_only",
    "message_and_diff",
    "git_show",
]


# Supports running the notebook either from results/rq3_1/analysis/
# or from the repository root.
DATA_PATH = Path("../metrics/macro_f1.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("results/rq3_1/metrics/macro_f1.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find macro_f1.csv."
    )


df = pd.read_csv(DATA_PATH)

if SUBJECT not in df.columns:
    raise ValueError(
        f"Required project column '{SUBJECT}' was not found."
    )

if df[SUBJECT].duplicated().any():
    duplicates = (
        df.loc[df[SUBJECT].duplicated(), SUBJECT]
        .astype(str)
        .tolist()
    )

    raise ValueError(
        f"Duplicate project identifiers were found: {duplicates}"
    )


missing_treatments = [
    treatment
    for treatment in TREATMENTS
    if treatment not in df.columns
]

if missing_treatments:
    raise ValueError(
        "Missing required input representations: "
        f"{missing_treatments}"
    )


for treatment in TREATMENTS:
    numeric = pd.to_numeric(
        df[treatment],
        errors="coerce",
    )

    observed = df[treatment].notna()

    if (
        observed.any()
        and not numeric.loc[observed].notna().all()
    ):
        raise ValueError(
            f"Treatment '{treatment}' contains non-numeric values."
        )

    df[treatment] = numeric


incomplete_mask = (
    df[TREATMENTS]
    .isna()
    .any(axis=1)
)

removed_projects = (
    df.loc[incomplete_mask, SUBJECT]
    .astype(str)
    .tolist()
)

df = df.loc[~incomplete_mask].copy()

if df[SUBJECT].nunique() < 3:
    raise ValueError(
        "Fewer than three complete paired projects remain."
    )


print(f"Data source: {DATA_PATH}")
print(f"Projects retained: {df[SUBJECT].nunique()}")
print(
    f"Input representations ({len(TREATMENTS)}): "
    f"{TREATMENTS}"
)

if removed_projects:
    print(
        "Projects removed by complete-case filtering: "
        f"{removed_projects}"
    )

display(df[[SUBJECT, *TREATMENTS]].head())


def _round_small(frame, decimals):
    """Round for display; nonzero values that round to 0 are shown as <0.001."""
    rounded = frame.round(decimals)
    out = rounded.astype(object)
    for column in frame.select_dtypes("number").columns:
        tiny = (frame[column] != 0) & (rounded[column] == 0)
        out.loc[tiny, column] = "<0.001"
    return out


Data source: ..\metrics\macro_f1.csv
Projects retained: 30
Input representations (4): ['message_only', 'diff_only', 'message_and_diff', 'git_show']


,project,message_only,diff_only,message_and_diff,git_show
0,cargo/clap,0.448229,0.559784,0.635528,0.632440
1,go/google-cloud-go,0.421964,0.369348,0.456164,0.456711
2,npm/angular,0.373868,0.318808,0.336685,0.336819
3,npm/ant-design,0.386911,0.332034,0.402581,0.428439
4,npm/biome,0.375830,0.463776,0.520424,0.464437


## Analysis functions

In [2]:
from itertools import combinations
from scipy.stats import friedmanchisquare


def _format_p_plain(p_value, digits=4):
    p_value = float(p_value)

    if p_value < 0.001:
        return "<.001"

    return f"{p_value:.{digits}f}".lstrip("0")


def _run_parametric_route(
    long_data,
    analysis_data,
    subject,
    treatments,
    dv_name,
    alpha,
):
    n_subjects = analysis_data[subject].nunique()
    k = len(treatments)

    sphericity = pg.sphericity(
        data=long_data,
        dv=dv_name,
        within="treatment",
        subject=subject,
    )
    apply_gg = bool(sphericity.pval < alpha)

    anova = pg.rm_anova(
        data=long_data,
        dv=dv_name,
        within="treatment",
        subject=subject,
        correction=True,
        detailed=True,
    )
    effect = anova.iloc[0]

    epsilon = float(effect["eps"])
    df1_uncorrected = k - 1
    df2_uncorrected = (k - 1) * (n_subjects - 1)

    if apply_gg:
        final_p = float(effect["p_GG_corr"])
        final_df1 = epsilon * df1_uncorrected
        final_df2 = epsilon * df2_uncorrected
        correction = "Greenhouse-Geisser"
    else:
        final_p = float(effect["p_unc"])
        final_df1 = float(df1_uncorrected)
        final_df2 = float(df2_uncorrected)
        correction = "None"

    omnibus = {
        "test": "Repeated-measures ANOVA",
        "statistic_name": "F",
        "statistic": float(effect["F"]),
        "df1": final_df1,
        "df2": final_df2,
        "p": final_p,
        "effect_name": "eta_G_squared",
        "effect": float(effect["ng2"]),
        "epsilon": epsilon,
        "correction": correction,
        "significant": final_p < alpha,
    }

    posthoc = None
    if final_p < alpha:
        posthoc = pg.pairwise_tests(
            data=long_data,
            dv=dv_name,
            within="treatment",
            subject=subject,
            parametric=True,
            padjust="holm",
            effsize="hedges",
            return_desc=True,
        )
        posthoc = posthoc[[
            "A", "B", "mean_A", "mean_B", "T", "dof",
            "p_unc", "p_corr", "p_adjust", "hedges",
        ]].copy()
        posthoc["significant"] = posthoc["p_corr"] < alpha

    return {
        "sphericity": sphericity,
        "anova": anova,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def _run_nonparametric_route(
    analysis_data,
    treatments,
    alpha,
):
    n_subjects = len(analysis_data)
    k = len(treatments)

    friedman = friedmanchisquare(
        *[analysis_data[treatment].to_numpy() for treatment in treatments]
    )
    q_statistic = float(friedman.statistic)
    p_value = float(friedman.pvalue)
    kendalls_w = q_statistic / (n_subjects * (k - 1))

    omnibus = {
        "test": "Friedman",
        "statistic_name": "Q",
        "statistic": q_statistic,
        "df1": float(k - 1),
        "df2": None,
        "p": p_value,
        "effect_name": "Kendalls_W",
        "effect": kendalls_w,
        "epsilon": None,
        "correction": "Not applicable",
        "significant": p_value < alpha,
    }

    posthoc = None
    if p_value < alpha:
        rows = []
        for treatment_a, treatment_b in combinations(treatments, 2):
            values_a = analysis_data[treatment_a].to_numpy()
            values_b = analysis_data[treatment_b].to_numpy()

            if np.allclose(values_a - values_b, 0):
                w_value = 0.0
                pair_p = 1.0
                rbc = 0.0
            else:
                wilcoxon = pg.wilcoxon(
                    values_a,
                    values_b,
                    alternative="two-sided",
                ).iloc[0]
                w_value = float(wilcoxon["W-val"])
                pair_p = float(wilcoxon["p-val"])
                rbc = float(wilcoxon["RBC"])

            rows.append({
                "A": treatment_a,
                "B": treatment_b,
                "mean_A": float(np.mean(values_a)),
                "mean_B": float(np.mean(values_b)),
                "W": w_value,
                "p_unc": pair_p,
                "RBC": rbc,
            })

        posthoc = pd.DataFrame(rows)
        reject, corrected_p = pg.multicomp(
            posthoc["p_unc"].to_numpy(),
            alpha=alpha,
            method="holm",
        )
        posthoc["p_corr"] = corrected_p
        posthoc["significant"] = reject

    return {
        "sphericity": None,
        "anova": None,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def _create_pairwise_table(posthoc, route):
    """Build a table with every pairwise comparison.

    The treatment with the higher mean is always listed first, so the
    test statistic and effect size are reported as magnitudes: direction is
    already encoded by the comparison order. Unlike `significant_posthoc`,
    this table is not filtered by significance — every pair produced by the
    post-hoc procedure is included, with a `Significant` column flagging
    which comparisons crossed alpha after Holm correction.
    """
    if posthoc is None:
        return None

    rows = []
    for _, row in posthoc.iterrows():
        if row["mean_A"] >= row["mean_B"]:
            better, worse = row["A"], row["B"]
        else:
            better, worse = row["B"], row["A"]

        if route == "parametric":
            statistic_name = "t"
            statistic = abs(float(row["T"]))
            effect_name = "g_av"
            effect = abs(float(row["hedges"]))
        else:
            statistic_name = "W"
            statistic = float(row["W"])
            effect_name = "r_rb"
            effect = abs(float(row["RBC"]))

        rows.append({
            "Comparison": f"{better} vs. {worse}",
            "Better": better,
            "Worse": worse,
            "Statistic name": statistic_name,
            "Statistic": statistic,
            "p_raw": float(row["p_unc"]),
            "p_Holm": float(row["p_corr"]),
            "Effect name": effect_name,
            "Effect": effect,
            "Significant": bool(row["significant"]),
        })

    # Significant comparisons first, then ordered by effect magnitude.
    return (
        pd.DataFrame(rows)
        .sort_values(["Significant", "Effect"], ascending=[False, False])
        .reset_index(drop=True)
    )


def run_automatic_repeated_measures_analysis(
    data,
    subject,
    treatments,
    dv_name="macro_f1",
    alpha=0.05,
):
    """Select and run the prespecified repeated-measures analysis.

    IMPORTANT: all pairwise comparisons are retained and displayed,
    including non-significant ones — not only the significant subset.
    """

    if len(treatments) < 3:
        raise ValueError("At least three treatments are required.")

    analysis_data = data[[subject, *treatments]].dropna().copy()
    long_data = analysis_data.melt(
        id_vars=subject,
        value_vars=treatments,
        var_name="treatment",
        value_name=dv_name,
    )

    descriptive = (
        long_data.groupby("treatment")[dv_name]
        .agg(N="count", Mean="mean", SD="std")
        .sort_values("Mean", ascending=False)
    )

    shapiro_rows = []
    for treatment in treatments:
        normality = pg.normality(
            analysis_data[treatment],
            method="shapiro",
            alpha=alpha,
        ).iloc[0]
        shapiro_rows.append({
            "Treatment": treatment,
            "N": int(analysis_data[treatment].notna().sum()),
            "W": float(normality["W"]),
            "p": float(normality["pval"]),
            "Normality not rejected": bool(normality["normal"]),
        })

    shapiro = pd.DataFrame(shapiro_rows).set_index("Treatment")
    all_normal = bool(shapiro["Normality not rejected"].all())

    if all_normal:
        route = "parametric"
        inferential = _run_parametric_route(
            long_data=long_data,
            analysis_data=analysis_data,
            subject=subject,
            treatments=treatments,
            dv_name=dv_name,
            alpha=alpha,
        )
    else:
        route = "non-parametric"
        inferential = _run_nonparametric_route(
            analysis_data=analysis_data,
            treatments=treatments,
            alpha=alpha,
        )

    posthoc = inferential["posthoc"]
    significant_posthoc = None
    pairwise_table = None
    if posthoc is not None:
        significant_posthoc = posthoc.loc[posthoc["significant"]].copy()
        pairwise_table = _create_pairwise_table(posthoc, route)

    print("DESCRIPTIVE STATISTICS")
    display(_round_small(descriptive, {"Mean": 4, "SD": 4}))


    print("SHAPIRO-WILK TESTS")

    shapiro_display = shapiro.copy()
    shapiro_display["W"] = shapiro_display["W"].round(4)
    shapiro_display["p"] = shapiro_display["p"].map(
        _format_p_plain
    )

    display(shapiro_display)


    print(f"SELECTED ROUTE: {route}")

    if inferential["sphericity"] is not None:
        sphericity = inferential["sphericity"]

        sphericity_display = pd.DataFrame([{
            "Mauchly W": round(sphericity.W, 4),
            "chi-square": round(sphericity.chi2, 4),
            "df": sphericity.dof,
            "p": _format_p_plain(sphericity.pval),
            "Sphericity": bool(sphericity.spher),
        }])

        display(sphericity_display)


    omnibus_display = pd.DataFrame([
        inferential["omnibus"]
    ]).copy()

    omnibus_display["p"] = omnibus_display["p"].map(
        _format_p_plain
    )

    display(_round_small(omnibus_display, {
        "statistic": 4,
        "df1": 4,
        "df2": 4,
        "effect": 4,
        "epsilon": 4,
    }))


    if posthoc is None:
        pass

    else:
        print(f"ALL HOLM-CORRECTED PAIRWISE COMPARISONS ({len(posthoc)} total)")

        all_posthoc_display = posthoc.copy()
        all_posthoc_display["p_unc"] = (
            all_posthoc_display["p_unc"].map(_format_p_plain)
        )
        all_posthoc_display["p_corr"] = (
            all_posthoc_display["p_corr"].map(_format_p_plain)
        )

        display(_round_small(all_posthoc_display, {
            "mean_A": 4,
            "mean_B": 4,
            "T": 4,
            "dof": 1,
            "hedges": 4,
            "W": 4,
            "RBC": 4,
        }))

        print("PAIRWISE TABLE")

        pairwise_display = pairwise_table.copy()
        pairwise_display["p_raw"] = pairwise_display["p_raw"].map(_format_p_plain)
        pairwise_display["p_Holm"] = pairwise_display["p_Holm"].map(_format_p_plain)

        display(_round_small(pairwise_display, {
            "Statistic": 4,
            "Effect": 4,
        }))

        print(
            "SIGNIFICANT HOLM-CORRECTED COMPARISONS "
            f"({len(significant_posthoc)} of {len(posthoc)})"
        )

        if significant_posthoc.empty:
            pass

        else:
            significant_display = significant_posthoc.copy()

            significant_display["p_unc"] = (
                significant_display["p_unc"].map(_format_p_plain)
            )
            significant_display["p_corr"] = (
                significant_display["p_corr"].map(_format_p_plain)
            )

            display(_round_small(significant_display, {
                "mean_A": 4,
                "mean_B": 4,
                "T": 4,
                "dof": 1,
                "hedges": 4,
                "W": 4,
                "RBC": 4,
            }))

    return {
        "route": route,
        "analysis_data": analysis_data,
        "long_data": long_data,
        "descriptive": descriptive,
        "shapiro": shapiro,
        "sphericity": inferential["sphericity"],
        "anova": inferential["anova"],
        "omnibus": inferential["omnibus"],
        "posthoc": posthoc,
        "significant_posthoc": significant_posthoc,
        "pairwise_table": pairwise_table,
        "alpha": alpha,
    }


## Run the analysis

In [3]:
analysis_results = run_automatic_repeated_measures_analysis(
    data=df,
    subject=SUBJECT,
    treatments=TREATMENTS,
)


DESCRIPTIVE STATISTICS


,N,Mean,SD
treatment,,,
git_show,30,0.4852,0.0971
message_and_diff,30,0.4834,0.1091
diff_only,30,0.4423,0.0918
message_only,30,0.4165,0.0953


SHAPIRO-WILK TESTS


,N,W,p,Normality not rejected
Treatment,,,,
message_only,30,0.9924,.9983,True
diff_only,30,0.9307,.0514,True
message_and_diff,30,0.9486,.1550,True
git_show,30,0.9496,.1650,True


SELECTED ROUTE: parametric


...

,A,B,mean_A,mean_B,T,dof,p_unc,p_corr,p_adjust,hedges,significant
0,diff_only,git_show,0.4423,0.4852,-4.6482,29.0,<.001,<.001,holm,-0.4484,True
1,diff_only,message_and_diff,0.4423,0.4834,-4.0015,29.0,<.001,.0012,holm,-0.4021,True
2,diff_only,message_only,0.4423,0.4165,1.862,29.0,.0728,.1455,holm,0.2722,False
3,git_show,message_and_diff,0.4852,0.4834,0.2469,29.0,.8067,.8067,holm,0.0176,False
4,git_show,message_only,0.4852,0.4165,6.6821,29.0,<.001,<.001,holm,0.7051,True
5,message_and_diff,message_only,0.4834,0.4165,6.0001,29.0,<.001,<.001,holm,0.6444,True


PAIRWISE TABLE


,Comparison,Better,Worse,Statistic name,Statistic,p_raw,p_Holm,Effect name,Effect,Significant
0,git_show vs. message_only,git_show,message_only,t,6.6821,<.001,<.001,g_av,0.7051,True
1,message_and_diff vs. message_only,message_and_diff,message_only,t,6.0001,<.001,<.001,g_av,0.6444,True
2,git_show vs. diff_only,git_show,diff_only,t,4.6482,<.001,<.001,g_av,0.4484,True
3,message_and_diff vs. diff_only,message_and_diff,diff_only,t,4.0015,<.001,.0012,g_av,0.4021,True
4,diff_only vs. message_only,diff_only,message_only,t,1.862,.0728,.1455,g_av,0.2722,False
5,git_show vs. message_and_diff,git_show,message_and_diff,t,0.2469,.8067,.8067,g_av,0.0176,False


SIGNIFICANT HOLM-CORRECTED COMPARISONS (4 of 6)


,A,B,mean_A,mean_B,T,dof,p_unc,p_corr,p_adjust,hedges,significant
0,diff_only,git_show,0.4423,0.4852,-4.6482,29.0,<.001,<.001,holm,-0.4484,True
1,diff_only,message_and_diff,0.4423,0.4834,-4.0015,29.0,<.001,.0012,holm,-0.4021,True
4,git_show,message_only,0.4852,0.4165,6.6821,29.0,<.001,<.001,holm,0.7051,True
5,message_and_diff,message_only,0.4834,0.4165,6.0001,29.0,<.001,<.001,holm,0.6444,True
